In [ ]:
import os
import pandas as pd
from datetime import datetime 
import duckdb
import unicodedata
import sys
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[05/07/26 16:17:48] INFO     Using                                                                  ]8;id=717806;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=97831;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framework\project\__init__.py#269\269]8;;\
                             'c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framewo                
                             rk\project\rich_logging.yml' as logging configuration.                                

[05/07/26 16:17:49] WARNING  c:\Users\User\miniconda3\envs\central\Lib\site-packages\requests\__ini ]8;id=931241;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py\warnings.py]8;;\:]8;id=187030;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py#110\110]8;;\
                             t__.py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet                     
                             (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported                    
                             version!                                                                              
                               warnings.warn(                                                                      
                                                                                                                   

[05/07/26 16:17:50] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=285540;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=665687;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\CENTRAL\central-perm-flow


In [ ]:
# Add the 'src' directory to the path
sys.path.append(os.path.abspath("src"))
import central_perm_flow.pipelines.data_processing.nodes as nodes_dproc
import central_perm_flow.pipelines.calac_activos_baj_grad.nodes as nodes_abg

# Importación de Fuentes

In [ ]:
central_calaca_ext  = catalog.load('central_calendario_extendido')
central_calaca_uptoday  = catalog.load('central_calendario_extendido_uptoday')
central_estaca_sd = catalog.load('central_estaca_sd')

[05/07/26 16:17:57] INFO     Loading data from central_calendario_extendido                    ]8;id=628537;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=388690;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_calendario_extendido_uptoday            ]8;id=373087;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=279875;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_estaca_sd (ParquetDataset)...           ]8;id=191628;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=380242;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [ ]:
central_estaca_sd.columns


Index(['identificacion', 'codigo_sis', 'id_inconcert', 'nombres',
       'usuario_institucional', 'alianza', 'cohorte', 'fecha_ingreso',
       'fecha_de_registro', 'nivel', 'nivel_academico', 'programa', 'estado',
       'fecha_de_baja_t', 'fecha_de_baja_d', 'tipo_baja', 'motivo_baja',
       'submotivo_baja', 'comentarios', 'fecha_de_reingreso', 'fecha_grado',
       'exito_estudiantil', 'etapa_studen_journey', 'descuentos', 'di', 'gi'],
      dtype='object')

In [ ]:
central_estaca_sd.loc[:, 'estado'].value_counts()


estado
activo                  1443
baja definitiva          154
egresado no graduado     148
baja temporal             58
Name: count, dtype: int64

## Importación de parámetros

In [ ]:
parameters = catalog.load("parameters")

In [ ]:
# bajas_calac
bajas_calac = parameters['bajas_calac']
graduados_calac = parameters['graduados_calac']

## Bajas

In [ ]:
bajas_calendario_academico = nodes_abg.momento_baja(
    # 1. El DataFrame (Objeto en memoria)
    central_estaca_sd=central_estaca_sd, 
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    fallback_weeks=parameters['graduados_calac']['graduation_fallback_weeks'],
    
    # 2. Valores extraídos de diccionarios de configuración
    central_col_fechadef=bajas_calac['central_col_fechadef'],
    central_col_fechatemp=parameters['bajas_calac']['central_col_fechatemp'],
    
    # 3. Nombre del dataset según el catálogo
    central_calaca= central_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 4. Parámetros de transformación y cruce
    left_on=parameters['bajas_calac']['merge_left_on'],
    right_on=parameters['bajas_calac']['merge_right_on'],
    group_key=parameters['bajas_calac']['central_calaca_col_cohorte_inicial'],
    sort_cols=parameters['bajas_calac']['central_calaca_col_sort']
)

In [ ]:
parameters['bajas_calac']['central_calaca_col_cohorte_inicial']

In [ ]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_desercion = (
    bajas_calendario_academico
    .loc[:, ['cohorte_inicial','nivel_academico', 'fecha_ingreso', 'semana_acumulada', 'di']]
    .groupby(['cohorte_inicial','nivel_academico',  'fecha_ingreso', 'semana_acumulada'])
    .agg({'di': 'sum'})
    .reset_index()
    .sort_values(by='di', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_desercion.style.bar(
    subset=['di'], 
    color='#f8766d', # Un rojo suave tipo "soft red"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

In [ ]:
bajas_calendario_academico = catalog.load('central_bajas_calendario_academico')

## Graduados

In [ ]:
graduados_calendario_academico = nodes_abg.momento_grado(
    # 1. Referencias a datasets/DataFrames
    central_estaca=central_estaca_sd,
    central_calaca=central_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 2. Configuración de lógica de negocio (Duraciones y Grado Inmediato)
   
    col_gi=parameters['graduados_calac']['graduation_col_gi'],
    
    # 3. Parámetro de seguridad (Semanas por defecto si falla el cálculo)
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    fallback_weeks=parameters['graduados_calac']['graduation_fallback_weeks'],
    
    # 4. Llaves para el cruce de tablas (Joins)
    join_left=parameters['graduados_calac']['graduation_join_keys_left'],
    join_right=parameters['graduados_calac']['graduation_join_keys_right']
)

In [ ]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_graduacion = (
    graduados_calendario_academico
    .loc[:, ['cohorte_inicial','nivel_academico','nivel', 'fecha_ingreso', 'semana_acumulada', 'gi']]
    .groupby(['cohorte_inicial','nivel_academico','nivel',  'fecha_ingreso', 'semana_acumulada'])
    .agg({'gi': 'sum'})
    .reset_index()
    .sort_values(by='gi', ascending=False)
    .head(100)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_graduacion.style.bar(
    subset=['gi'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

## Activos

In [ ]:
activos_calendario_academico = nodes_abg.momento_activos(
   central_estaca=central_estaca_sd,
    central_calaca=central_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 2. Configuración de lógica de negocio (Duraciones y Grado Inmediato)
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    col_di='di',
    col_gi='gi',
    
    # 3. Parámetro de seguridad (Semanas por defecto si falla el cálculo)
    fallback_weeks=None,
    
    # 4. Llaves para el cruce de tablas (Joins)
    join_left='fecha_activo',
    join_right='fecha_inicio',
    group_key = 'fecha_ingreso'
) 

In [ ]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_eng = (
    activos_calendario_academico
    .loc[:, ['cohorte_inicial','nivel_academico','nivel','programa', 'fecha_ingreso', 'semana_acumulada','max_semana_teorica', 'engi']]
    .groupby(['cohorte_inicial','nivel_academico','nivel','programa',  'fecha_ingreso', 'semana_acumulada','max_semana_teorica'])
    .agg({'engi': 'sum'})
    .reset_index()
    .sort_values(by='engi', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_eng.style.bar(
    subset=['engi'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

In [ ]:
top_diez_activos = (
    activos_calendario_academico
    .groupby(['cohorte_inicial','fecha_ingreso','nivel_academico','nivel','programa',  'semana_acumulada','max_semana_teorica'])
    .agg(activos=('ai', 'sum')) # <-- Definimos: nombre = (columna, operacion)
    .reset_index()
    .sort_values(by='activos', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_activos.style.bar(
    subset=['activos'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

In [ ]:
def consolidar_universo_academico(
    df_bajas: pd.DataFrame,
    df_graduados: pd.DataFrame,
    df_activos: pd.DataFrame
) -> pd.DataFrame:
    """
    Concatena los tres estados académicos y normaliza indicadores.
    """
    # Unimos los tres dataframes
    universo = pd.concat([df_bajas, df_graduados, df_activos], ignore_index=True)
    
    # Normalizamos la columna engi: si es nula (viene de bajas/grados), es 0
    if 'engi' in universo.columns:
        universo['engi'] = universo['engi'].fillna(0).astype(int)
        
    return universo

In [ ]:
df_bajas = catalog.load('central_bajas_calendario_academico')
df_graduados = catalog.load('central_graduados_calendario_academico')
df_activos = catalog.load('central_activos_calendario')

In [ ]:
df_consolidado = catalog.load('central_estados_calac')

In [ ]:
central_estados_perm = catalog.load('central_estados_calac')

central_estados_perm = central_estados_perm.copy()

In [ ]:
top_diez_activos = (
    df_consolidado
    .groupby(['cohorte_inicial','fecha_ingreso'])
    .agg(estudiantes=('identificacion', 'count')) # <-- Definimos: nombre = (columna, operacion)
    .reset_index()
    .sort_values(by='estudiantes', ascending=False)
  
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_activos.style.bar(
    subset=['estudiantes'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook


In [ ]:
top_diez_activos.sort_values(by= 'fecha_ingreso').head(20)

In [ ]:
central_estados_perm['programa']

In [ ]:
central_estaca = catalog.load('central_estaca')

In [ ]:
central_estaca['CENTRAL'].columns

In [ ]:
central_estaca['CENTRAL'].shape

In [ ]:
central_estaca['CENTRAL'].groupby(['Cohorte']).agg(estudiantes=('Identificación', 'count'))

In [ ]:
df_consolidado = catalog.load('central_estados_calac')

In [ ]:
df_consolidado.groupby(['cohorte_inicial','fecha_ingreso']).agg({'ai':'sum',
                                                 'di':'sum',
                                                 'gi':'sum',
                                                 'engi':'sum',})

In [ ]:
df_consolidado.columns

In [ ]:
df_consolidado.loc[:,['identificacion', 'codigo_sis', 'nombres',
       'usuario_institucional', 'alianza', 'cohorte', 'cohorte_inicial',
       'cohorte_actual', 'nivel', 'nivel_academico', 'programa', 'estado',
       'bloque', 'descuentos', 'fecha_ingreso', 'fecha_de_registro',
       'fecha_de_baja_t', 'fecha_de_baja_d', 'fecha_baja',
       'fecha_de_reingreso', 'fecha_grado', 'fecha_activo', 'tipo_baja',
       'motivo_baja', 'submotivo_baja', 'comentarios', 'periodo_raw']]

# Solicitudes

## Activos

In [ ]:
central_activos_calendario = catalog.load('central_activos_calendario')
# Pregrado
mask_pregrado = central_activos_calendario['nivel_academico'] == 'pregrado'
central_activos_calendario_pregrado = central_activos_calendario[mask_pregrado]
# Posgrado
mask_posgrado = central_activos_calendario['nivel_academico'] == 'posgrado'
central_activos_calendario_posgrado = central_activos_calendario[mask_posgrado]

In [ ]:
comunicaciones_col = ['id_inconcert','nombres','identificacion','usuario_institucional','programa','cohorte_actual']

In [ ]:
central_activos_calendario_pregrado_comunicaciones = central_activos_calendario_pregrado.loc[:,comunicaciones_col].rename(columns = {'id_inconcert':'ID Inconcert',
                                                                                              'nombres':'Nombre Completo',
                                                                                              'identificacion': 'Documento de identidad',
                                                                                              'usuario_institucional':'Correo electrónico institucional',
                                                                                              'programa':'Programa académico',
                                                                                              'cohorte_actual':'Periodo Academico'})
central_activos_calendario_pregrado_comunicaciones['Telefono'] = None
central_activos_calendario_pregrado_comunicaciones['Correo electrónico personal'] = None

In [ ]:
central_activos_calendario_prosgrado_comunicaciones = central_activos_calendario_posgrado.loc[:,comunicaciones_col].rename(columns = {'id_inconcert':'ID Inconcert',
                                                                                              'nombres':'Nombre Completo',
                                                                                              'identificacion': 'Documento de identidad',
                                                                                              'usuario_institucional':'Correo electrónico institucional',
                                                                                              'programa':'Programa académico',
                                                                                              'cohorte_actual':'Periodo Academico'})
central_activos_calendario_prosgrado_comunicaciones['Telefono'] = None
central_activos_calendario_prosgrado_comunicaciones['Correo electrónico personal'] = None

In [ ]:
comunicaciones_col_output = ['ID Inconcert', 'Nombre Completo', 'Documento de identidad',
                        'Correo electrónico personal','Correo electrónico institucional','Telefono', 'Programa académico',
                        'Periodo Academico']

# Pregrado

In [ ]:
central_activos_calendario_pregrado_comunicaciones.shape

In [ ]:
central_activos_calendario_pregrado_comunicaciones.loc[:,comunicaciones_col_output].to_excel('data/07_model_output/01_solicitudes/central_activos_pregrado.xlsx', index = False)

# Posgrado

In [ ]:
central_activos_calendario_prosgrado_comunicaciones.loc[:,comunicaciones_col_output].to_excel('data/07_model_output/01_solicitudes/central_activos_prosgrado.xlsx', index = False)

In [ ]:
central_activos_calendario_prosgrado_comunicaciones.shape

In [ ]:
central_bajas_calendario_academico = catalog.load('central_bajas_calendario_academico')

In [ ]:
mask_cohorte = central_bajas_calendario_academico['cohorte'] == pd.to_datetime('2025-09-01')
mask_programa = central_bajas_calendario_academico['programa'] == 'ingenieria de sistemas y computacion'

In [ ]:
central_bajas_calendario_academico.loc[mask_cohorte & mask_programa,:]

In [ ]:
mask_cohorte = central_activos_calendario['cohorte'] == pd.to_datetime('2025-09-01')
mask_programa = central_activos_calendario['programa'] == 'ingenieria de sistemas y computacion'

In [ ]:
central_activos_calendario[mask_cohorte & mask_programa]

# Recupero = Bajas

In [3]:
central_bajas_calendario_academico = catalog.load('central_bajas_calendario_academico')

[05/05/26 15:48:48] INFO     Loading data from central_bajas_calendario_academico              ]8;id=863564;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=379264;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

In [5]:
central_bajas_calendario_academico.shape

(196, 43)

In [ ]:
central_bajas_calendario_academico.columns


Index(['identificacion', 'codigo_sis', 'id_inconcert', 'nombres',
       'usuario_institucional', 'alianza', 'cohorte', 'fecha_ingreso',
       'fecha_de_registro', 'nivel', 'nivel_academico', 'programa', 'estado',
       'fecha_de_baja_t', 'fecha_de_baja_d', 'tipo_baja', 'motivo_baja',
       'submotivo_baja', 'comentarios', 'fecha_de_reingreso', 'fecha_grado',
       'exito_estudiantil', 'etapa_studen_journey', 'descuentos', 'di', 'gi',
       'fecha_baja', 'max_semana_teorica', 'periodo_raw', 'cohorte_inicial',
       'cohorte_actual', 'bloque', 'fecha_inicio', 'fecha_fin',
       'shifted_fecha_inicio', 'semana', 'semana_acumulada', 'month',
       'mes_academico', 'anio_gregoriano', 'mes_gregoriano', 'student_journey',
       'tipo'],
      dtype='object')

In [26]:
recupero_cohortes = ['2025 3a','2025 1e','2025 2c']
mask_cohorte = central_bajas_calendario_academico['cohorte_actual'].isin(recupero_cohortes)
mask_cohorte.sum()

np.int64(40)

In [28]:
central_bajas_calendario_academico_recuperos = central_bajas_calendario_academico[mask_cohorte]

In [30]:
documentos = [1012450394,
1016026483,
1065643508,
35897196,
1022958687,
79046862,
1007840184]

In [37]:
central_estados_calac = catalog.load('central_estados_calac')

[05/05/26 16:02:43] INFO     Loading data from central_estados_calac (ExcelDataset)...         ]8;id=588413;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=461196;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [43]:
central_estados_calac.loc[central_estados_calac['identificacion'].isin(documentos), ['identificacion','nombres','cohorte', 'cohorte_inicial','estado','programa','fecha_de_baja_t', 'fecha_de_baja_d', 'fecha_baja']]

,identificacion,nombres,cohorte,cohorte_inicial,estado,programa,fecha_de_baja_t,fecha_de_baja_d,fecha_baja
138,1016026483,diana rocio franco cortes,2025-10-27,2025 1e,baja definitiva,contaduria publica,NaT,2026-04-07,2026-04-07
187,1012450394,triana romero anyi carolina,2026-03-16,2026 1b,baja definitiva,ciencias tributarias,NaT,2026-03-17,2026-03-17
444,1022958687,javier samuel leon rodriguez,2026-01-19,2026 1a,activo,ingenieria de sistemas y computacion,NaT,NaT,NaT
845,79046862,mario ivan castiblanco castellanos,2026-03-16,2026 1b,activo,derecho,NaT,NaT,NaT
924,35897196,murillo barrios nirama,2026-03-16,2026 1b,activo,auditoria y control,NaT,NaT,NaT


In [ ]:
central_bajas_calendario_academico.loc[:,['cohorte_actual']].value_counts().reset_index().sort_values(by = 'cohorte_actual')